# Blaze 编程范式：用组件组装矩阵乘算子

> **新手提示**：本节介绍 Blaze 的核心思想——像搭积木一样组装算子。读完后你将理解"为什么用 Blaze 更简单"。

本节学习大纲：
- 为什么需要 Blaze——传统开发的痛点 vs Blaze 的解法
- Tensor-Layout 编程范式——数据怎么存、怎么搬
- Blaze 整体架构——组件分工清晰
- 组装范式——5 步完成矩阵乘

---



## 0. 前置知识：矩阵乘硬件基础

### 0.1 问题：为什么不能直接计算大矩阵？

假设计算 C[1024,1024] = A[1024,512] × B[512,1024]
- 数据量：A 约 1MB + B 约 1MB + C 约 2MB（FP16）
- **问题**：NPU 的计算单元（L0）只有几十 KB，无法一次性放入所有数据

### 0.2 Ascend NPU 内存层级（图解）

```text
┌─────────────────────────────────────────┐
│  GM (Global Memory) - 几百MB~几GB       │  ← 数据存储在大内存
│    ↓ CopyGM2L1                          │
│  L1 (L1 Buffer) - 约 1MB                │  ← 中转缓存
│    ↓ CopyL12L0A/B                       │
│  L0A/L0B - 几十KB                        │  ← CubeCore 输入
│    ↓ Mmad (矩阵乘计算)                  │
│  L0C - 几十KB (float)                    │  ← 计算结果
│    ↓ CopyL0C2GM                         │
│  GM                                      │  ← 写回结果
└─────────────────────────────────────────┘
```

**关键理解**：
- GM 能存大量数据，但计算必须把数据搬到 L0
- L0A/L0B/L0C 很小，所以必须**分块计算**（把大矩阵切成小块）
- 一个 **Tile** = 一块能放入 L0 的小数据块

### 0.3 分块计算示意

```text
大矩阵 1024×1024  →  切成 16×16 个小块（每块 64×64）

┌────┬────┬────┬────┐
│Tile│Tile│Tile│Tile│  每个小块独立计算
├────┼────┼────┼────┤  最后拼成完整结果
│... │... │... │... │
└────┴────┴────┴────┘
```

---



## 1. 为什么需要 Blaze

### 1.1 传统方式：手写搬运代码

传统 Ascend C 开发需要手动处理：

<table style="text-align: left; margin-left: 0">
<tr><th align="left">要做的事</th><th align="left">传统方式</th><th align="left">痛点</th></tr>
<tr><td align="left">数据搬运</td><td align="left">手动计算偏移地址</td><td align="left">容易出错，代码冗长</td></tr>
<tr><td align="left">分块调度</td><td align="left">手动写循环逻辑</td><td align="left">复杂，难以复用</td></tr>
<tr><td align="left">适配不同数据类型</td><td align="left">为 FP16/BF16/INT8 分别写代码</td><td align="left">代码重复，维护困难</td></tr>
<tr><td align="left">适配不同布局</td><td align="left">为 ND/NZ/ZN 分别写代码</td><td align="left">变体爆炸</td></tr>
</table>

**组合爆炸**：数据类型 × 布局格式 × 优化策略 = 数百种变体，每种都要写一遍！

### 1.2 Blaze 的解法：像搭积木

**类比**：
- 传统方式 = 手工画流程图，每个场景重画
- Blaze = 选择积木块 → 拼装 → 自动生成代码

Blaze 的核心思路：

<table style="text-align: left; margin-left: 0">
<tr><th align="left">特性</th><th align="left">说明</th><th align="left">新手理解</th></tr>
<tr><td align="left">编译期参数化</td><td align="left">类型/布局作为模板参数</td><td align="left">编译器自动生成对应代码</td></tr>
<tr><td align="left">策略驱动派发</td><td align="left">DispatchPolicy 决定流水线</td><td align="left">选择策略，自动选最优路径</td></tr>
<tr><td align="left">分层架构</td><td align="left">Kernel/Block/Epilogue/Policy</td><td align="left">组件分工清晰，可复用</td></tr>
<tr><td align="left">Tensor-Layout</td><td align="left">Layout/Shape/Coord 简化计算</td><td align="left">不需要手动算偏移</td></tr>
</table>

**简单理解**：你只需要告诉 Blaze "我要什么"，它会自动处理 "怎么搬运、怎么计算"。

## 2. Tensor-Layout 编程范式

### 2.1 Tensor = 数据 + 元信息

**类比**：Tensor 就像快递包裹
- 包裹内容 = 实际数据（矩阵元素）
- 包裹标签 = 元信息（形状、布局、类型）

任何 Tensor 都需三步构建：

```cpp
// Step 1: 数据在哪（内存地址）
auto memPtr = MakeMemPtr<Location::GM>(地址);

// Step 2: 数据怎么排（布局）
auto layout = MakeFrameLayout<ND>(Shape{m, n});

// Step 3: 粘合成 Tensor
auto tensor = MakeTensor(memPtr, layout);
```

<img src="./images/01.03/tensor_layout.png" alt="Tensor-Layout构建" width="800px">

### 2.2 Tensor 的好处

**好处**：用 Tensor 切片时，不需要手动计算偏移！

```cpp
// 传统方式：手动算偏移（复杂、易错）
// offset = row * n + col
// data[offset] = ...

// Tensor 方式：一行代码切片
auto block = tensor.Slice(Coord{0, 0}, Shape{64, 64});
// Blaze 自动帮你算偏移！
```

**类比**：传统方式像手动计算地址，Tensor 方式像 GPS 导航——直接输入坐标，自动计算路径。

### 2.3 四种 Layout（布局）

布局决定了数据在内存中的排列方式：

<table style="text-align: left; margin-left: 0">
<tr><th align="left">布局</th><th align="left">排列方式</th><th align="left">常见场景</th></tr>
<tr><td align="left">ND（Row-major）/ DN (Colum-major)</td><td align="left">按行排 [0,1,2,3...] / 按列排 </td><td align="left">训练场景最常见</td></tr>
<tr><td align="left">NZ/ZN（分形）</td><td align="left">按块排</td><td align="left">推理优化场景</td></tr>
</table>

**新手推荐**：入门阶段用 **ND** 就够了！

注意： 实际上Blaze中表示ND布局，用的类型为 `NDExtLayoutPtn` 

---



## 3. Blaze 整体架构

<img src="./images/01.03/blaze_layer_architecture.png" alt="Blaze四层架构" width="800px">

### 3.1 类比理解

Blaze核心分层架构为： Kernel + BlockScheduler + BlockMmad + BlockEpilogue, 可类比工厂流水线如下理解：

```text
┌─────────────────────────────────────────┐
│ Kernel（工厂主管）                       │
│   - 接收任务参数                         │
│   - 分配任务给工人                       │
├─────────────────────────────────────────┤
│ BlockScheduler（调度员）                 │
│   - 把大矩阵切成小块                     │
│   - 分配给不同核                         │
├─────────────────────────────────────────┤
│ BlockMmad（工人）                        │
│   - 执行实际计算                         │
│   - 搬运数据 → 计算 → 输出               │
├─────────────────────────────────────────┤
│ BlockEpilogue（质检员）                  │
│   - 后处理（累加、激活）                 │
│   - Basic 场景可能不需要                 │
└─────────────────────────────────────────┘
```

### 3.2 核心层级职责总结

<table style="text-align: left; margin-left: 0">
<tr><th align="left">层级</th><th align="left">角色</th><th align="left">职责</th><th align="left">新手理解</th></tr>
<tr><td align="left">Kernel</td><td align="left">主管</td><td align="left">组合 Block + Epilogue + Scheduler</td><td align="left">算子入口</td></tr>
<tr><td align="left">BlockScheduler</td><td align="left">调度员</td><td align="left">切分矩阵、分配任务</td><td align="left">切蛋糕</td></tr>
<tr><td align="left">BlockMmad</td><td align="left">工人</td><td align="left">搬运 + MMAD 计算 + 输出</td><td align="left">干活</td></tr>
<tr><td align="left">BlockEpilogue</td><td align="left">质检员</td><td align="left">后处理（累加、激活）</td><td align="left">检验</td></tr>
<tr><td align="left">Policy</td><td align="left">策略</td><td align="left">定义调度策略、常量</td><td align="left">选择方案</td></tr>
</table>

**关键理解**：每个组件职责清晰，你只需要"选择组件 → 组装"。



## 4. DispatchPolicy——策略选择

### 4.1 什么是 DispatchPolicy？

**类比**：DispatchPolicy 是"方案选择器"。
- 选择不同的 DispatchPolicy = 选择不同的实现方案
- 编译器会自动生成对应的最优代码

<img src="./images/01.03/dispatch_policy.png" alt="DispatchPolicy派发机制" width="700px">

### 4.2 当前支持策略

<table style="text-align: left; margin-left: 0">
<tr><th align="left">策略</th><th align="left">适用场景</th><th align="left">新手理解</th></tr>
<tr><td align="left">MatmulMultiBlockBasic</td><td align="left">基本矩阵乘</td><td align="left">入门首选</td></tr>
<tr><td align="left">MatmulMultiBlockWithStreamK</td><td align="left">StreamK 矩阵乘</td><td align="left">K轴切分并行</td></tr>
<tr><td align="left">MatmulWithScaleMx</td><td align="left">MX 格式量化</td><td align="left">MX 量化</td></tr>
</table>

**新手推荐**：入门阶段用 **MatmulMultiBlockBasic<0, 0>** 就够了！



## 5. 组装范式——5 步完成矩阵乘

### 5.1 Blaze 的核心使用流程

**5 步组装**：选择策略 → 定义类型 → 组合组件 → 组装 Kernel → 执行

### 5.2 BF16 Basic 示例（完整流程）

```cpp
// ===== Step 1: 选择 DispatchPolicy =====
using Policy = Blaze::Gemm::MatmulMultiBlockBasic<0, 0>;  // Basic 策略
// 参数说明：<0, 0> 表示：不全载、无融合

// ===== Step 2: 定义数据类型和布局 =====
using AType     = bfloat16_t;                        // A 矩阵用 BF16
using BType     = bfloat16_t;                        // B 矩阵用 BF16
using CType     = bfloat16_t;                        // C 矩阵用 BF16
using BiasType  = bfloat16_t;                        // Bias 用 BF16
using LayoutA   = AscendC::Te::NDExtLayoutPtn;       // A: ND 布局
using LayoutB   = AscendC::Te::NDExtLayoutPtn;       // B: ND 布局
using LayoutC   = AscendC::Te::NDExtLayoutPtn;       // C: ND 布局
using Shape     = AscendC::Te::Shape<int64_t, int64_t, int64_t, int64_t>; // (m,n,k,batch)

// ===== Step 3: 组合 Block 级组件 =====
using Scheduler = Blaze::Gemm::Block::BlockSchedulerMatmulBasic<Shape>;  // 调度器
using Mmad      = Blaze::Gemm::Block::BlockMmad<                         // 计算组件
                    Policy, AType, LayoutA, BType, LayoutB, CType, LayoutC, BiasType, LayoutC>;
using Epilogue  = Blaze::Gemm::Block::BlockEpilogueEmpty;                 // 后处理：Basic用Empty

// ===== Step 4: 组装 Kernel =====
using Kernel = Blaze::Gemm::Kernel::GemmUniversal<Shape, Mmad, Epilogue, Scheduler>;

// ===== Step 5: 构造参数并执行 =====
using Params = typename Kernel::Params;
Params params = {
    {m, n, k, batch},                        // 矩阵维度
    {aGM, bGM, cGM, biasGM},                 // A/B/C/Bias 的 GM 地址
    {},                                      // epilogueParams（Empty为空）
    {mL1, nL1, kL1, baseM, baseN, baseK, ...} // 分块参数
};

Kernel mm;
mm(params);  // 一行代码完成矩阵乘！
```

完整组装流程如下图所示：

<img src="./images/01.03/kernel_arch.png" alt="Kernel组装架构" width="700px">

### 5.3 策略简介

| 选项 | 是什么 |
| :--- | :--- |
| **MatmulMultiBlockBasic<0,0>** | 最朴素的多 Block 分发：SwizzleOffset=0、SwizzleDir=0，即 row-major 顺序把 MN 切给各个 AI Core，**无 pingpong、无 ShuffleK、无 StreamK、无量化融合**，逻辑最干净 |
| **MatmulMultiBlockWithStreamK** | 在 MultiBlock 基础上加 StreamK 并行（K 方向也能切跨 Core），用于负载不均衡的大 K 场景 |
| **MatmulWithScaleMx** | Microscaling (MXFP8/MXFP4)，A5(DAV_3510) 新精度，scaleA/scaleB 按 K/32 block 共享，入门不会先碰 |
| **MatmulWithScaleFixpipeQuant** | 量化 Matmul + Fixpipe 随路量化融合，W8A8/FP8 推理专用，A2 起才有 Fixpipe 硬化 |

### 5.4 关键理解

<table style="text-align: left; margin-left: 0">
<tr><th align="left">步骤</th><th align="left">你做什么</th><th align="left">Blaze 做什么</th></tr>
<tr><td align="left">1</td><td align="left">选择策略</td><td align="left">根据策略选择代码路径</td></tr>
<tr><td align="left">2</td><td align="left">定义类型</td><td align="left">生成对应类型的搬运/计算代码</td></tr>
<tr><td align="left">3</td><td align="left">组合组件</td><td align="left">组装完整流水线</td></tr>
<tr><td align="left">4</td><td align="left">组装 Kernel</td><td align="left">创建算子入口</td></tr>
<tr><td align="left">5</td><td align="left">执行</td><td align="left">自动完成搬运、计算、输出</td></tr>
</table>

**新手提示**：你只需要关心 Step 1-4 的"选择"和"定义"，Step 5 的执行是自动的！

> **下一节预告**：[01.04 Blaze 接口介绍](./01.04_blaze_api_introduction.ipynb) 详细介绍每个组件的参数；[01.05 Basic MatMul 实操](./01.05_blaze_basic_matmul.ipynb) 完整落地上述 5 步流程。

---



## 6. 课后练习

1. **（判断题）** Blaze 的 DispatchPolicy 在编译期选择代码路径，而不是运行期分支。

2. **（判断题）** Tensor 方式需要手动计算偏移地址。

3. **（单选题）** Blaze 的哪个组件负责"切分矩阵、分配任务"？
   - A. Kernel
   - B. BlockScheduler
   - C. BlockMmad
   - D. BlockEpilogue

4. **（单选题）** 入门阶段推荐用哪个 DispatchPolicy？
   - A. MatmulMultiBlockWithStreamK
   - B. MatmulMultiBlockBasic<0, 0>
   - C. MatmulWithScaleMx
   - D. MatmulWithScaleFixpipeQuant

5. **（填空题）** Blaze 组装算子的 5 步流程为：选择 ______ → 定义 ______ → 组合组件 → 组装 Kernel → 执行。

**执行以下代码获取答案。**


In [ ]:
!cat ./answer/01.03_answer.txt